In [1]:
'''Gemma 4 Developer Agent | Single agent, bounded shell search

Configuration reference: https://github.com/happyc0der/gemma-swe-agent
Reference commit: ba5b80fe77b0ed018af919fb148977c1504623c0.
The workflow prompt below is our own implementation, informed by the public baseline.
It corrects early submission and tool argument advice; this is not an exact replay.
This candidate has no measured leaderboard score. The Pathfinder adaptation scored 0.08.
The Roman adaptation scored 0.06; the previous single-agent candidate failed evaluation.
This revision removes search_similar_code after reported context overflow incidents.
It does not establish or fix the unknown cause of the previous hosted exception.
Run all cells to create submission.zip. No model inference is performed here.
'''

from pathlib import Path
import hashlib
import importlib.util
import os
import shutil
import zipfile
import yaml

OUTPUT_DIR = Path(os.environ.get('GEMMA_OUTPUT_ROOT', '/kaggle/working'))
if not OUTPUT_DIR.is_dir() and 'GEMMA_OUTPUT_ROOT' not in os.environ:
    OUTPUT_DIR = Path.cwd() / 'gemma4_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_DIR = OUTPUT_DIR / 'gemma4_shell_search_bundle'
ZIP_PATH = OUTPUT_DIR / 'submission.zip'

In [2]:
'''Keep generation and task limits explicit. Setup time is additional to agent time.'''

MODEL = 'gemma-4-31b-it-qat-w4a16-ct'
SAMPLING = {
    'temperature': 0.2,
    'top_p': 0.95,
    'top_k': 40,
    'max_output_tokens': 4096,
    'thinking_config': {'include_thoughts': False},
}
EVALUATION = {
    'timeout_seconds': 180,
    'max_tool_calls': 40,
    'max_time_minutes': 4.5,
    'max_turns': 80,
}
TOOLS = ['run_command', 'edit_file', 'write_file', 'get_status',
         'submit_patch', 'get_code_neighbors']

'''These are planning scenarios, not measurements or a guarantee about hidden task count.
Verification is excluded from the competition's agent budget.
'''
ASSUMED_TASK_COUNT = 120
ASSUMED_SETUP_MINUTES_PER_TASK = 1.0
ASSUMED_FIXED_OVERHEAD_MINUTES = 15.0
estimated_minutes = (
    ASSUMED_TASK_COUNT * (EVALUATION['max_time_minutes'] + ASSUMED_SETUP_MINUTES_PER_TASK)
    + ASSUMED_FIXED_OVERHEAD_MINUTES
)
print(f'Planning scenario: {estimated_minutes / 60:.2f} hours; '
      f'{720 - estimated_minutes:.0f} minutes below the 12-hour cap.')
print('Replace these assumptions with measurements on the current harness and 4x L4.')

Planning scenario: 11.25 hours; 45 minutes below the 12-hour cap.
Replace these assumptions with measurements on the current harness and 4x L4.


In [3]:
'''Implement localization, a minimal patch, bounded verification, and final submission.'''

SYSTEM_PROMPT = '''You are a Python software engineer resolving the repository issue in the user message. Work in /workspace. Your goal is a correct implementation patch that passes the issue's hidden tests.

Workspace rules
- Change implementation files only. Preserve public behavior outside the requested fix.
- Read existing tests for expected behavior and conventions, but never alter tests, conftest.py, pytest.ini, or CI files to hide failures.
- Put temporary reproduction and editing scripts in /tmp using run_command. write_file creates or overwrites files inside /workspace; use it only for intended source additions.
- Dependencies are prepared and the sandbox is offline. Do not install packages.

Locate and understand
1. Extract symbols, file names, error messages, and expected behavior from the issue. Search concrete identifiers first with bounded grep output.
2. Read the relevant function, its callers when relevant, and the nearest existing tests. Use run_command with sed to read at most 80 lines per call. All command outputs should be bounded.
3. Localize source with shell search: grep -R -n -F --include="*.py" -- "symbol" candidate_directory | head -n 30. Search likely directories first; exclude environments, generated files, and dependency folders. Read only relevant regions with sed. Use get_code_neighbors only when a known symbol needs caller or dependency context; if it cannot resolve the symbol, continue with shell search.
4. Form a specific root-cause hypothesis and patch plan. Avoid exhaustive repository browsing. Aim for 4-6 search calls and 3-5 bounded reads, adjusting to the remaining time.

Reproduce and fix
5. When practical, create a short reproduction in /tmp with a quoted shell heredoc and execute it once before editing. If setup is difficult, use the nearest existing focused test instead. Do not spend the entire task preparing a reproduction.
6. Apply a small patch with edit_file. Pass filepath, old_string, and new_string with real newlines and exact indentation copied from the source. Avoid whole-file rewrites.
7. If old_string is not found, read the exact region and retry with a smaller unique snippet. After a second failure, use a /tmp Python script that asserts the replacement count is one before writing the source. Never repeat an identical failed call.
8. Handle the edge cases named in the issue and check relevant callers. Keep the patch focused; do not refactor unrelated code.

Verify and submit
9. Rerun the reproduction and run only the nearest existing tests, with short output and an explicit timeout appropriate to the remaining time. For example, timeout 45s python3 -m pytest tests/test_target.py -x -q --tb=short. A timed-out test is inconclusive, not a passing test. Do not run the entire test suite.
10. Correct regressions caused by your patch. Check git diff and git status --short; remove only unintended artifacts that you created.
11. Call submit_patch after verification, or with your best available patch when time is nearly exhausted. Treat submit_patch as final: the harness can end the agent loop after the current turn. Do not plan additional verification after submitting.

Execution budget
- Keep reasoning concise and spend time on actual tool actions.
- Use get_status at the beginning, after approximately 5 ordinary tool calls, before tests, and before submission. It reports the shared task budget; extra agents do not create extra time.
- Aim to finish within approximately 25 ordinary tool calls, but use the live time and call allowance as the actual limits.
- With less than 60 seconds or fewer than 5 ordinary calls left, stop further exploration. Complete a small justified patch, run one short check if time allows, inspect the diff, and submit before the budget expires.
- Avoid commands that could consume the remaining session. Reproduction and tests should normally have a shell timeout of 20-45 seconds, reduced when less time remains.
- If the same approach fails twice, change your approach. Do not repeatedly rerun a reproduction that already answered the question.
- Use documented argument types: command, filepath, old_string, new_string, query, and node are strings; optional k and max_neighbors are integers, and allow_multiple is boolean. Omit optional arguments unless necessary.
'''

In [4]:
'''Build only the files needed by the single-agent submission.'''

AGENT = {
    'name': 'gemma4_single_agent',
    'model': MODEL,
    'description': 'Resolve repository issues with focused patches and bounded verification.',
    'instruction': '!include prompts/system.md',
    'generate_content_config': '!include configs/sampling.yaml',
    'tools': TOOLS,
}
agent_yaml = yaml.safe_dump(AGENT, sort_keys=False)
agent_yaml = agent_yaml.replace("'!include prompts/system.md'", '!include prompts/system.md')
agent_yaml = agent_yaml.replace("'!include configs/sampling.yaml'", '!include configs/sampling.yaml')
FILES = {
    'agent.yaml': agent_yaml,
    'configs/sampling.yaml': yaml.safe_dump(SAMPLING, sort_keys=False),
    'eval_config.yaml': yaml.safe_dump({'evaluation': EVALUATION}, sort_keys=False),
    'prompts/system.md': SYSTEM_PROMPT,
    'NOTICE.txt': (
        'Configuration and workflow reference: happyc0der/gemma-swe-agent\n'
        'Commit: ba5b80fe77b0ed018af919fb148977c1504623c0\n'
        'Our prompt implements the discussed workflow with corrected submission semantics.\n'
        'No leaderboard score has been measured for this candidate.\n'
    ),
}
shutil.rmtree(BUNDLE_DIR, ignore_errors=True)
for relative, content in FILES.items():
    path = BUNDLE_DIR / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.rstrip() + '\n', encoding='utf-8')
print('Prepared:', BUNDLE_DIR)

Prepared: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/gemma-4-developer-agent/artifacts/shell_search/gemma4_shell_search_bundle


In [5]:
'''Validate YAML, include resolution and task limits. Use the installed official schema if available.'''

class IncludeLoader(yaml.SafeLoader):
    pass

def load_include(loader, node):
    relative = loader.construct_scalar(node)
    assert not Path(relative).is_absolute() and '..' not in Path(relative).parts, relative
    target = (Path(loader.name).parent / relative).resolve()
    assert BUNDLE_DIR.resolve() in target.parents and target.is_file(), target
    if target.suffix in ('.yaml', '.yml'):
        with target.open(encoding='utf-8') as stream:
            return yaml.load(stream, Loader=IncludeLoader)
    return target.read_text(encoding='utf-8')

IncludeLoader.add_constructor('!include', load_include)
with (BUNDLE_DIR / 'agent.yaml').open(encoding='utf-8') as stream:
    resolved_agent = yaml.load(stream, Loader=IncludeLoader)
with (BUNDLE_DIR / 'eval_config.yaml').open(encoding='utf-8') as stream:
    resolved_eval = yaml.safe_load(stream)
assert resolved_agent['model'] == MODEL
assert resolved_agent['instruction'] == SYSTEM_PROMPT.rstrip() + '\n'
assert resolved_agent['generate_content_config'] == SAMPLING
assert resolved_agent['tools'] == TOOLS and all(isinstance(t, str) for t in TOOLS)
assert 'sub_agents' not in resolved_agent and 'submit_patch' in TOOLS
assert 'search_similar_code' not in TOOLS
assert resolved_eval == {'evaluation': EVALUATION}
assert set(EVALUATION) == {'timeout_seconds', 'max_tool_calls', 'max_time_minutes', 'max_turns'}
assert 0 < EVALUATION['timeout_seconds'] <= EVALUATION['max_time_minutes'] * 60
assert EVALUATION['max_tool_calls'] > 0 and EVALUATION['max_turns'] > 0
assert '{' not in SYSTEM_PROMPT and '}' not in SYSTEM_PROMPT

if importlib.util.find_spec('adk_submission'):
    from adk_submission import validate_directory
    from adk_submission.schema import SandboxedAgentConfig
    validate_directory(BUNDLE_DIR)
    SandboxedAgentConfig.model_validate(resolved_agent)
    print('Installed official directory and agent schema checks passed.')
else:
    print('Official package unavailable: current Kaggle compiler/scorer check still required.')
print('YAML, includes, task limits and single-agent structure passed.')

Official package unavailable: current Kaggle compiler/scorer check still required.
YAML, includes, task limits and single-agent structure passed.


In [6]:
'''Create a deterministic ZIP with configs at its root. Packaging is not a model evaluation.'''

with zipfile.ZipFile(ZIP_PATH, 'w') as archive:
    for relative in sorted(FILES):
        entry = zipfile.ZipInfo(relative, date_time=(1980, 1, 1, 0, 0, 0))
        entry.compress_type = zipfile.ZIP_DEFLATED
        entry.external_attr = 0o100644 << 16
        archive.writestr(entry, (BUNDLE_DIR / relative).read_bytes())
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(FILES)
    assert yaml.safe_load(archive.read('eval_config.yaml')) == {'evaluation': EVALUATION}
print('Submission:', ZIP_PATH)
print('SHA-256:', hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
print('Archive files:', sorted(FILES))
print('No 4x L4 development evaluation or leaderboard submission has been run.')

Submission: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/gemma-4-developer-agent/artifacts/shell_search/submission.zip
SHA-256: dce57d4c7a2e61393bb26ee1ff41720b55efd1fd49e1a5c01883211ab30d3b53
Archive files: ['NOTICE.txt', 'agent.yaml', 'configs/sampling.yaml', 'eval_config.yaml', 'prompts/system.md']
No 4x L4 development evaluation or leaderboard submission has been run.
